In [ ]:
!pip install --upgrade google-cloud-bigquery

In [ ]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import numpy as np
from scipy import stats

In [ ]:
auth.authenticate_user()

In [ ]:
client = bigquery.Client(project="data_analytics-mate")

In [ ]:
query = """
with session_info as(
  SELECT
  s.date,
  s.ga_session_id,
  sp.country,
  sp.device,
  sp.continent,
  sp.channel,
  ab.test,
  ab.test_group,
  from `DA.ab_test` as ab
  join `DA.session` as s
  on ab.ga_session_id = s.ga_session_id
  join `DA.session_params` as sp
  on sp.ga_session_id = ab.ga_session_id
),
session_with_orders as(
Select
  session_info.date,
  session_info.country,
  session_info.device,
  session_info.continent,
  session_info.channel,
  session_info.test,
  session_info.test_group,
  count(DISTINCT o.ga_session_id) as session_with_orders
from `DA.order`as o
join session_info
on o.ga_session_id = session_info.ga_session_id
GROUP BY
  session_info.date,
  session_info.country,
  session_info.device,
  session_info.continent,
  session_info.channel,
  session_info.test,
  session_info.test_group
),
events as (SELECT
session_info.date,
  session_info.country,
  session_info.device,
  session_info.continent,
  session_info.channel,
  session_info.test,
  session_info.test_group,
  ep.event_name,
  count(ep.ga_session_id) as event_cnt
from `DA.event_params` as ep
join session_info
on ep.ga_session_id = session_info.ga_session_id
group by
session_info.date,
  session_info.country,
  session_info.device,
  session_info.continent,
  session_info.channel,
  session_info.test,
  session_info.test_group,
  ep.event_name),
session as (select
session_info.date,
  session_info.country,
  session_info.device,
  session_info.continent,
  session_info.channel,
  session_info.test,
  session_info.test_group,
  count(distinct session_info.ga_session_id) as session_cnt
from session_info
group by session_info.date,
  session_info.country,
  session_info.device,
  session_info.continent,
  session_info.channel,
  session_info.test,
  session_info.test_group),
account as(
Select session_info.date,
  session_info.country,
  session_info.device,
  session_info.continent,
  session_info.channel,
  session_info.test,
  session_info.test_group,
  count(distinct acs.ga_session_id) as new_account
  from `DA.account_session`as acs
  join session_info
  on acs.ga_session_id = session_info.ga_session_id
group by session_info.date,
  session_info.country,
  session_info.device,
  session_info.continent,
  session_info.channel,
  session_info.test,
  session_info.test_group)


SELECT session_with_orders.date,
  session_with_orders.country,
  session_with_orders.device,
  session_with_orders.continent,
  session_with_orders.channel,
  session_with_orders.test,
  session_with_orders.test_group,
  'session_with_orders' as event_name,
  session_with_orders.session_with_orders as value
FROM session_with_orders
UNION ALL
SELECT
events.date,
  events.country,
  events.device,
  events.continent,
  events.channel,
  events.test,
  events.test_group,
  event_name,
  events.event_cnt as value
FROM events
UNION ALL
SELECT
session.date,
  session.country,
  session.device,
  session.continent,
  session.channel,
  session.test,
  session.test_group,
  "session" as event_name,
  session_cnt as value
FROM session
UNION ALL
SELECT
account.date,
  account.country,
  account.device,
  account.continent,
  account.channel,
  account.test,
  account.test_group,
  "new_account" as event_name,
  new_account as value
FROM account
"""

df = pd.read_gbq(query, project_id = "data-analytics-mate")

/tmp/ipykernel_3412/3378428234.py:148: FutureWarning: read_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.read_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.read_gbq
  df = pd.read_gbq(query, project_id = "data-analytics-mate")


To obtain data for analysis, we have to connect to Google BigQuery and retrieve the necessary dataset using an SQL query. A data frame has been created based on this dataset, which will be used for analysis.

In [ ]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   date        800996 non-null  dbdate
 1   country     800996 non-null  object
 2   device      800996 non-null  object
 3   continent   800996 non-null  object
 4   channel     800996 non-null  object
 5   test        800996 non-null  Int64 
 6   test_group  800996 non-null  Int64 
 7   event_name  800996 non-null  object
 8   value       800996 non-null  Int64 
dtypes: Int64(3), dbdate(1), object(5)
memory usage: 57.3+ MB
None


In [ ]:
print(df.head())

         date      country   device continent         channel  test  \
0  2020-11-01        Kenya  desktop    Africa  Organic Search     2   
1  2020-11-01  Puerto Rico  desktop  Americas  Organic Search     2   
2  2020-11-02      Bahrain  desktop      Asia       Undefined     2   
3  2020-11-02  New Zealand  desktop   Oceania  Organic Search     2   
4  2020-11-03      Croatia  desktop    Europe     Paid Search     2   

   test_group           event_name  value  
0           1  session_with_orders      1  
1           2  session_with_orders      1  
2           2  session_with_orders      1  
3           1  session_with_orders      1  
4           1  session_with_orders      1  


In [ ]:
#check which event names are in our dataset
print(df["event_name"].unique())

['session_with_orders' 'new_account' 'session' 'page_view' 'view_item'
 'user_engagement' 'session_start' 'first_visit' 'begin_checkout'
 'view_promotion' 'scroll' 'select_item' 'add_to_cart' 'add_shipping_info'
 'view_search_results' 'add_payment_info' 'select_promotion' 'click'
 'view_item_list']


In [ ]:
#creating the metrics dictionary. If we need to calculate a new metric we can simply add it here.
metrics = {
    "add_payment_info": "value",
    "add_shipping_info": "value",
    "begin_checkout": "value",
    "new_account": "value"
}

In [ ]:
#creating the pivot table for calculations
pivot = df.pivot_table(
    index = ["date", "country", "device", "continent", "channel", "test", "test_group"],
    columns = "event_name",
    values = "value",
    aggfunc = "sum"
).reset_index()

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

In [ ]:
#creating an empty list for results
results_list = []
#loop to determine the test number
for test_num in pivot["test"].unique():
    test_data = pivot[pivot["test"]==test_num]
#loop for calculating each previously defined event name between the control and test groups
    for metric in metrics:
        control_group = test_data[test_data["test_group"] == 1]
        test_group = test_data[test_data["test_group"] == 2]
#сalculating the conversion rate for event_name (calculated as a percentage of the total number of sessions)
        control_metric = control_group[metric].sum()
        test_metric = test_group[metric].sum(0)
        control_sessions = control_group["session"].sum()
        test_sessions = test_group["session"].sum()
        test_conversion = test_metric / test_sessions
        control_conversion = control_metric / control_sessions
        metric_change = (test_conversion - control_conversion) / control_conversion *100

#determine statistical significance using the proportion test (z-test)
        z_stat, p_value = proportions_ztest([test_metric, control_metric], [test_sessions, control_sessions])
        if p_value < 0.05:
          significant = True
        else:
          significant = False

#creating the dictionary with results and adding it to the empty results list
        results_list.append({
        "test_number": test_num,
        "metric": metric,
        "numerator_event": metric,
        "denominator_event": "session",
        "test_denominator": test_sessions,
        "test_conversion_rate": test_conversion,
        "ctrl_numerator_count": control_metric,
        "ctrl_denominator_conversion": control_sessions,
        "ctrl_conversion_rate": control_conversion,
        "metric_change": metric_change,
        "z-statistic": z_stat,
        "p-value": p_value,
        "significant": significant
  })

In [ ]:
#creating the dataframe from the lists with results
results_df = pd.DataFrame(results_list)
results_df.tail(6)

,test_number,metric,numerator_event,denominator_event,test_denominator,test_conversion_rate,ctrl_numerator_count,ctrl_denominator_conversion,ctrl_conversion_rate,metric_change,z-statistic,p-value,significant
10,3,begin_checkout,begin_checkout,session,70439,0.131518,9532,70047,0.136080,-3.352445,-2.511389,0.012026,True
11,3,new_account,new_account,session,70439,0.082653,5856,70047,0.083601,-1.133880,-0.643489,0.519907,False
12,4,add_payment_info,add_payment_info,session,105141,0.034249,3731,105079,0.035507,-3.541234,-1.571106,0.116158,False
13,4,add_shipping_info,add_shipping_info,session,105141,0.047137,5128,105079,0.048801,-3.411125,-1.785795,0.074132,False
14,4,begin_checkout,begin_checkout,session,105141,0.116672,12555,105079,0.119482,-2.351523,-1.995998,0.045934,True
15,4,new_account,new_account,session,105141,0.082622,8984,105079,0.085498,-3.362896,-2.375457,0.017527,True


In [ ]:
#saving results in csv-file
results_df.to_csv("results.csv", header = True)

[A link to a CSV file containing the calculation results.](https://drive.google.com/file/d/1pdOmwvfDf_NF1j4nczVKbjjoOh895v50/view?usp=sharing)


[The dashboard with results of A/B-testing.](https://public.tableau.com/views/AB-testingwiththedeterminationofthestatisticalsignificance/AB-testingwiththedeterminationofthestatisticalsignificance?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)

The results of the first A/B test show a statistically significant increase in conversion rates for the test version across the following metrics:
- add_payment_info by 12.12%;
- add_shipping_info by 6.16%;
- begin_checkout by 6.26%.

The Test No. 2 did not yield statistically significant results for any of the metrics, but the interim results show a positive trend. Therefore, if we increase the sample size, there is a likelihood that we will obtain a statistically significant positive result.

The Test No. 3 shows a statistically significant decrease in the begin_checkout conversion rate by 2.81%. The other metrics also show a negative trend, although without proven statistical significance. In this case, it’s better not to wait for the test to finish and to stop it.

The Test No. 4 showed a statistically significant decrease in conversion rates:
- begin_checkout by 2.29%;
- new_account by 3.31%.

Overall, Test No. 1 has statistically significant positive results. Therefore, the changes tested in the B version of this test can be implemented in the main version going forward.